# Use environment variables to select execution parameters for assessing Data Quality

In [1]:
import os

os.chdir("../../../")

In [2]:
print(os.getcwd())

D:\Python\Project_Boardgames_Data_Validation


In [3]:
os.environ['DQ_MODE'] = 'incremental'
os.environ['DQ_DATASOURCE'] = 'games_bgg'
os.environ['DQ_LAYER'] = 'bronze'
os.environ['DQ_LIMIT'] = '100'

- Initialize the Spark connection
- Read data based on the selected parameters

In [4]:
from data_quality.gx.plugins.connectors import SparkConnector

connector = SparkConnector()
connector.connect()
dtlk_df = connector.read_data()

In [5]:
datasource = connector.datasource
dtlk_df_layer = connector.layer

dq_suite = f"{dtlk_df_layer}_{datasource}_suite"
dq_asset = f"{dtlk_df_layer}_{datasource}_asset"

In [6]:
dq_suite, dq_asset

('bronze_games_bgg_suite', 'bronze_games_bgg_asset')

In [24]:
# dtlk_df.select('ingestion_timestamp').show(3,truncate=False)
dtlk_df.show(3)

+----+--------------------+-------+--------------------+-----------+-----------+--------+--------+--------+----+----------+-----------+---------+--------------------+---+--------------------+-----+--------------------+--------------------+------+--------------------+-------------+
|rank|             bgg_url|game_id|               names|min_players|max_players|avg_time|min_time|max_time|year|avg_rating|geek_rating|num_votes|           image_url|age|            mechanic|owned|            category|            designer|weight| ingestion_timestamp|source_system|
+----+--------------------+-------+--------------------+-----------+-----------+--------+--------+--------+----+----------+-----------+---------+--------------------+---+--------------------+-----+--------------------+--------------------+------+--------------------+-------------+
|   1|https://boardgame...| 174430|          Gloomhaven|          1|          4|     120|      60|     120|2017|   8.98893|    8.61858|    15376|https://c

In [8]:
# os.chdir("data_quality/")

In [9]:
import great_expectations as gx
context = gx.get_context(project_root_dir="./gx")

In [10]:
context.add_or_update_expectation_suite(expectation_suite_name=dq_suite)

{
  "expectation_suite_name": "bronze_games_bgg_suite",
  "ge_cloud_id": null,
  "expectations": [],
  "data_asset_type": null,
  "meta": {
    "great_expectations_version": "0.18.22"
  }
}

In [11]:
dataframe_datasource = context.sources.add_or_update_spark(
    name=f"{dtlk_df_layer}_{datasource}",
)

In [12]:
dq_asset = dataframe_datasource.add_dataframe_asset(
    name=dq_asset,
    dataframe=dtlk_df,
)

In [13]:
dq_request = dq_asset.build_batch_request()

In [14]:
dq_validator = context.get_validator(
    batch_request=dq_request,
    expectation_suite_name=dq_suite,
)

In [15]:
dq_validator.interactive_evaluation = False

In [17]:
from data_quality.gx.plugins.metadata import METADATA

metadata = {"dimension": "Completeness", "Layer": dtlk_df_layer}

for attribute, attribute_info in METADATA[dtlk_df_layer][datasource]['model'].items():
    dq_validator.expect_column_to_exist(attribute, meta=metadata)
    if not attribute_info['nullable']:
        dq_validator.expect_column_values_to_not_be_null(attribute, meta=metadata)

dq_validator.expect_table_row_count_to_be_between(
    min_value=2,
    meta=metadata
)

dq_validator.expect_table_column_count_to_equal(
    value=22,
    meta=metadata
)

{
  "success": null,
  "result": {},
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

In [31]:
import datetime


In [32]:
min_date

99

In [20]:
import datetime

metadata = {"dimension": "Timeliness", "Layer": dtlk_df_layer}

now = datetime.datetime.now()
start_of_year = datetime.datetime(now.year, 1, 1)
days_back = (now - start_of_year).days + 1
min_date = now - datetime.timedelta(days=days_back)

dq_validator.expect_column_max_to_be_between(
    column='ingestion_timestamp',
    min_value=min_date,
    meta=metadata
)

{
  "success": null,
  "result": {},
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

In [26]:
metadata = {"dimension": "Validity", "Layer": dtlk_df_layer}

dq_validator.expect_table_row_count_to_be_between(
    column='ingestion_timestamp',
    regex='\d{4}-(0[1-9]|1[0-2])-(0[1-9]|[12][0-9]|3[01])\s([01][0-9]|2[0-3]):[0-5][0-9]:[0-5][0-9]\.\d+',
    meta=metadata
)

for attribute, attribute_info in METADATA[dtlk_df_layer][datasource]['model'].items():
    dq_validator.expect_column_values_to_be_of_type(
        column=attribute,
        type_=attribute_info['data_type'],
        meta=metadata
    )

In [25]:
metadata = {"dimension": "Uniqueness", "Layer": dtlk_df_layer}

dq_validator.expect_column_values_to_be_unique(
    column='rank',
    meta=metadata
)

{
  "success": null,
  "result": {},
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

In [28]:
dq_validator.save_expectation_suite(discard_failed_expectations=False)


checkpoint = context.add_or_update_checkpoint(
    name=f"{dtlk_df_layer}_{datasource}_checkpoint",
    run_name_template=f"%Y%m%d-%H%M%S-{dtlk_df_layer}-{datasource}",
    validations=[
        {
            "batch_request":dq_request,
            "expectation_suite_name":dq_suite,
        },
    ],
    action_list=[
        {
            "name": "store_validation_result",
            "action": {"class_name": "StoreValidationResultAction"}
        },
    ],
)

context.add_or_update_checkpoint(checkpoint=checkpoint)

checkpoint_result = checkpoint.run()
context.build_data_docs()
context.open_data_docs()

Calculating Metrics:   0%|          | 0/45 [00:00<?, ?it/s]

In [ ]:
# TBA